In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_excel('Downloads/Excel_sales_w04_2.xlsx')

In [3]:
df.head(3)

,Row.ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer.Name,Segment,City,State,...,Category,Sub.Category,Product.Name,Sales,Quantity,Discount,Profit,Shipping_Cost,Order_Priority,Return
0,32298,CA-2012-124891,7/31/2012,7/31/2012,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,...,Technology,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...,2309.650,7,0.0,762.1845,933.57,Critical,No
1,26341,IN-2013-77878,2013-05-02 00:00:00,2013-07-02 00:00:00,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,...,Furniture,Chairs,"Novimex Executive Leather Armchair, Black",3709.395,9,0.1,-288.7650,923.63,Critical,Yes
2,25330,IN-2013-71249,2012-06-10 00:00:00,10/18/2013,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,...,Technology,Phones,"Nokia Smart Phone, with Caller ID",5175.171,9,0.1,919.9710,915.49,Medium,No


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25002 entries, 0 to 25001
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row.ID          25002 non-null  int64  
 1   Order_ID        25002 non-null  object 
 2   Order_Date      25002 non-null  object 
 3   Ship_Date       25002 non-null  object 
 4   Ship_Mode       25002 non-null  object 
 5   Customer_ID     25002 non-null  object 
 6   Customer.Name   25002 non-null  object 
 7   Segment         24998 non-null  object 
 8   City            25002 non-null  object 
 9   State           25002 non-null  object 
 10  Country         25002 non-null  object 
 11  Postal.Code     4046 non-null   float64
 12  Market          25002 non-null  object 
 13  Region          25002 non-null  object 
 14  Product.ID      25002 non-null  object 
 15  Category        25002 non-null  object 
 16  Sub.Category    25002 non-null  object 
 17  Product.Name    25002 non-null 

In [5]:
df['Order_ID'].fillna(method='ffill', inplace=True)

In [6]:
#df.info()

In [7]:
from dateutil import parser

# Function to convert Excel date serial numbers to datetime
def convert_excel_date(serial_number):
    if pd.notnull(serial_number):
        try:
            return pd.to_datetime('1900-01-01') + pd.to_timedelta(serial_number, 'D')
        except ValueError:
            return pd.NaT
    else:
        return pd.NaT

# Function to parse various date formats
def parse_date(date_str):
    try:
        return parser.parse(str(date_str), fuzzy=True)
    except parser.ParserError:
        return pd.NaT

# Assuming 'Order_Date' is the column with different date formats
df['Order_DateN'] = df['Order_Date'].apply(lambda x: convert_excel_date(x) if isinstance(x, (int, float)) else parse_date(x))



# Extracting date-related information
df['Date'] = pd.to_datetime(df['Order_DateN'], errors='coerce', format='%m/%d/%Y')
df['Year'] = df['Order_DateN'].dt.year

# Handle non-finite values in Quarter
df['Quarter'] = df['Order_DateN'].dt.quarter
df['Quarter'] = df['Quarter'].replace([np.inf, -np.inf, np.nan], 0).astype(int)

df['Quarter_Name'] = df['Order_DateN'].dt.to_period('Q').astype(str).str.replace(r'\d+Q', 'Q')
# Handle non-finite values in Month
df['Month'] = df['Order_DateN'].dt.month.replace([np.inf, -np.inf, np.nan], 0).astype(int)

df['Month_Name'] = df['Order_DateN'].dt.strftime('%B')
df['Month_Name_Short'] = df['Order_DateN'].dt.strftime('%b')  # Short month names
df['Week_Number'] = df['Order_DateN'].dt.isocalendar().week
# Handle non-finite values in Weekday
df['Weekday'] = (df['Order_DateN'].dt.weekday + 1).replace([np.inf, -np.inf, np.nan], 0).astype(int)  # Adding 1 to make Monday = 1, Sunday = 7

df['Weekday_Name'] = df['Order_DateN'].dt.strftime('%A')
df['Weekday_Name_Short'] = df['Order_DateN'].dt.strftime('%a')  # Short weekday names

# Displaying the resulting table
df_dates_by_manish = df[['Order_Date','Order_DateN', 'Date', 'Year', 'Quarter', 'Quarter_Name', 'Month', 'Month_Name', 'Month_Name_Short', 'Week_Number', 'Weekday', 'Weekday_Name', 'Weekday_Name_Short']].head(50)
df_dates_by_manish

,Order_Date,Order_DateN,Date,Year,Quarter,Quarter_Name,Month,Month_Name,Month_Name_Short,Week_Number,Weekday,Weekday_Name,Weekday_Name_Short
0,7/31/2012,2012-07-31,2012-07-31,2012,3,2012Q3,7,July,Jul,31,2,Tuesday,Tue
1,2013-05-02 00:00:00,2013-05-02,2013-05-02,2013,2,2013Q2,5,May,May,18,4,Thursday,Thu
2,2012-06-10 00:00:00,2012-06-10,2012-06-10,2012,2,2012Q2,6,June,Jun,23,7,Sunday,Sun
3,2005-01-28 00:00:00,2005-01-28,2005-01-28,2005,1,2005Q1,1,January,Jan,4,5,Friday,Fri
4,2013-05-11 00:00:00,2013-05-11,2013-05-11,2013,2,2013Q2,5,May,May,19,6,Saturday,Sat
5,2013-06-28 00:00:00,2013-06-28,2013-06-28,2013,2,2013Q2,6,June,Jun,26,5,Friday,Fri
6,2011-07-11 00:00:00,2011-07-11,2011-07-11,2011,3,2011Q3,7,July,Jul,28,1,Monday,Mon
7,04/14/2012,2012-04-14,2012-04-14,2012,2,2012Q2,4,April,Apr,15,6,Saturday,Sat
8,10/14/2014,2014-10-14,2014-10-14,2014,4,2014Q4,10,October,Oct,42,2,Tuesday,Tue
9,2012-01-28 00:00:00,2012-01-28,2012-01-28,2012,1,2012Q1,1,January,Jan,4,6,Saturday,Sat


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25002 entries, 0 to 25001
Data columns (total 37 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Row.ID              25002 non-null  int64         
 1   Order_ID            25002 non-null  object        
 2   Order_Date          25002 non-null  object        
 3   Ship_Date           25002 non-null  object        
 4   Ship_Mode           25002 non-null  object        
 5   Customer_ID         25002 non-null  object        
 6   Customer.Name       25002 non-null  object        
 7   Segment             24998 non-null  object        
 8   City                25002 non-null  object        
 9   State               25002 non-null  object        
 10  Country             25002 non-null  object        
 11  Postal.Code         4046 non-null   float64       
 12  Market              25002 non-null  object        
 13  Region              25002 non-null  object    

In [9]:
from IPython.display import FileLink

# Assuming the DataFrame is named df_dates_by_manish
df_dates_by_manish.to_excel('df_dates_by_manish.xlsx', index=False)

# Displaying a download link
FileLink(r'df_dates_by_manish.xlsx')

C:\Users\pawan\df_dates_by_manish.xlsx